In [1]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving archive (4).zip to archive (4).zip
User uploaded file "archive (4).zip" with length 215934 bytes


# Extracting the Dataset
We will start by extracting the uploaded zip file (`archive (4).zip`) to access the underlying CSV data.

In [2]:
import zipfile
import os

with zipfile.ZipFile('archive (4).zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

print(os.listdir('dataset'))

['spam.csv']


# Loading and Exploring the Dataset
We load the CSV file into a pandas DataFrame. Since SMS spam datasets can have varying encodings, we try common encodings like `latin-1` and clean up any unnecessary columns.

In [3]:
import pandas as pd

df = pd.read_csv('dataset/spam.csv', encoding='latin-1')
df = df.dropna(how='any', axis=1)
df.columns = ['label', 'message']

display(df.head())
print(df['label'].value_counts())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


label
ham     4825
spam     747
Name: count, dtype: int64


# Data Preprocessing and Feature Extraction
We encode the target labels ('ham' as 0 and 'spam' as 1), split the dataset into training and testing sets, and convert the text messages into numerical features using TF-IDF Vectorization.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['message'],
    df['label_num'],
    test_size=0.2,
    random_state=42
)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Model Selection, Training and Evaluation
We train a Multinomial Naive Bayes classifier on the vectorized text data and evaluate its performance using a classification report, accuracy score, and confusion matrix.

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9668161434977578

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.75      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115


Confusion Matrix:
 [[965   0]
 [ 37 113]]


# Conclusion
In this task, we successfully built an AI model to detect spam SMS messages.
- **Data Handling:** We extracted and loaded the text dataset, cleaning up empty columns.
- **Feature Representation:** TF-IDF Vectorization was used to transform raw text into robust numerical representations.
- **Model Performance:** The Multinomial Naive Bayes classifier achieved high accuracy on the testing subset, demonstrating the effectiveness of machine learning in filtering unwanted spam communication.